## Expected-result anchors (training split only)
Numbers behind `docs/expected_results.md`, written before any model is run on 2023 so the
expectations can't be fitted to the results. Only 2014-2022 is used.

In [12]:
import sys
from pathlib import Path
import numpy as np
# Read the src module
repo_root = Path.cwd().parent
sys.path.append(str(repo_root))
import yaml
from src.data.clean import clean_data
from src.data.split import get_split
with open(repo_root / "config" / "config.yaml") as f:
    config = yaml.safe_load(f)

### 1. Load the training split
`y` = `min_delay`, the target we predict (minutes).

In [13]:
train = get_split(clean_data(config), config, "training_set")
y = train["min_delay"]

Rows in 850590
dropped 39
Rows out 850551
mapped to Unknown: 65930
direction
East       192053
West       185255
North      173049
South      155067
Both        79197
Unknown     65930
Name: count, dtype: int64
Rows in 850551
dropped (missing / non-numeric / outside 1 - 999): 2849
Rows out 847702
distinct routes: 503
Rows in 847702
dropped total (target filter): 38020
Rows out 809682
training_set: 2014-2022, rows 683475


### 2. Median vs mean as a constant prediction
The naive benchmark predicts one number for every incident; any model must beat it.
MAE = average minutes off (primary metric). RMSE squares errors, so big misses count more (secondary).

In [14]:
median = y.median()
mean = y.mean()
print(f"median: {median:.2f}")
print(f"mean:   {mean:.2f}")
print(f"MAE  predicting median: {(y - median).abs().mean():.2f}")
print(f"MAE  predicting mean:   {(y - mean).abs().mean():.2f}")
print(f"RMSE predicting mean:   {np.sqrt(((y - mean) ** 2).mean()):.2f}")
print(f"RMSE predicting median: {np.sqrt(((y - median) ** 2).mean()):.2f}")

median: 10.00
mean:   13.42
MAE  predicting median: 7.17
MAE  predicting mean:   8.01
RMSE predicting mean:   15.06
RMSE predicting median: 15.45


- Median 10 is the benchmark; it is off by **7.17 min** on average, the bar models must beat.
- Median wins on MAE (7.17 vs 8.01), mean wins on RMSE (15.06 vs 15.45), so the median is the fair benchmark for an MAE task.
- Linear Regression minimises squared error, so expect it to do better on RMSE than on MAE.

### 3. Tail and mode
The tail = the few very long delays that pull the mean above the median.

In [15]:
print(f"share > 30 min: {(y > 30).mean():.3f}")
print(f"share > 60 min: {(y > 60).mean():.3f}")
for mode in ["bus", "streetcar"]:
    y_mode = y[train["mode"] == mode]
    print(f"{mode}: median {y_mode.median():.1f}, mean {y_mode.mean():.1f}, rows {len(y_mode)}")

share > 30 min: 0.040
share > 60 min: 0.016
bus: median 10.0, mean 13.8, rows 569214
streetcar: median 7.0, mean 11.7, rows 114261


- 4% of delays are over 30 min and 1.6% over 60 min: this tail is why mean > median.
- Streetcar delays are shorter than bus (median 7 vs 10), so mode, and probably route, should help predictions.

### 4. Trend by year
Checks whether 2023 is likely to differ from the training years (distribution shift).

In [16]:
year = train["timestamp"].dt.year
y_2022 = y[year == 2022]
print(f"2022 rows, MAE predicting the overall training median: {(y_2022 - median).abs().mean():.2f}")
train.groupby(year)["min_delay"].agg(["median", "mean", "count"]).round(2)

2022 rows, MAE predicting the overall training median: 7.70


,median,mean,count
timestamp,,,
2014,10.0,12.30,103341
2015,10.0,12.52,86767
2016,10.0,12.74,89290
2017,10.0,12.88,82118
2018,10.0,13.07,86849
2019,10.0,13.85,71634
2020,10.0,14.58,41378
2021,10.0,15.26,50908
2022,10.0,15.65,71190


- The median stays at 10, but the mean rises from 12.3 to 15.7: long delays are getting longer.
- 2022 alone gives MAE 7.70, so we expect the benchmark's 2023 MAE to be around 7.7-8.5, not 7.17.